# 🛰️ LISS-IV Satellite Cloud Removal GAN: Google Colab Training Notebook
### Adapted Pix2Pix Conditional GAN (cGAN) for ISRO ResourceSat LISS-IV Imagery
- **Sensor**: ResourceSat-2 / 2A LISS-IV (5.8m Ground Resolution)
- **Spectral Bands**: Green ($B_2$), Red ($B_3$), Near-Infrared ($B_4$)
- **Architecture**: 4-Channel U-Net 256 Generator + $70 \times 70$ PatchGAN Discriminator
- **Loss Functions**: Adversarial Loss + $L_1$ Loss + Spectral Angle Mapper (SAM) + NDVI Consistency Loss

---
### ⚡ Instructions before starting:
1. Navigate to **Runtime** $\to$ **Change runtime type**.
2. Under **Hardware accelerator**, select **GPU** (T4).
3. Click **Save**.


## Step 1: Verify NVIDIA GPU & Environment


In [ ]:
from google.colab import drive
import os, glob, zipfile
from pathlib import Path

# 1. Mount Google Drive (if using Drive)
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'[*] Drive mount skipped or already mounted: {e}')

# 2. Automated Smart Dataset Locator & Unzipper
print('\n[*] Searching for your LISS-IV dataset...')

def locate_dataset():
    # A. Check if already unzipped anywhere in /content or Drive
    candidates = list(Path('/content').rglob('Clear_Patches')) + list(Path('/content/drive/MyDrive').rglob('Clear_Patches'))
    for c in candidates:
        patches = list(c.glob('*.npy'))
        if len(patches) > 0:
            cloudy = c.parent / 'Cloudy_Patches'
            if cloudy.exists() and len(list(cloudy.glob('*.npy'))) > 0:
                return c, cloudy

    # B. Look for zip archives in /content or Google Drive to auto-extract
    zip_candidates = list(Path('/content').glob('*.zip')) + list(Path('/content/drive/MyDrive').rglob('*.zip'))
    # Prioritize liss_iv zips
    zip_candidates.sort(key=lambda x: 'liss' not in x.name.lower())
    
    for z in zip_candidates:
        if 'sample_data' in str(z): continue
        print(f'[*] Found zip archive: {z}')
        print(f'[*] Auto-extracting {z.name} to /content/ ... (please wait)')
        with zipfile.ZipFile(z, 'r') as zf:
            zf.extractall('/content/')
        print('[OK] Extraction completed!')
        
        # Re-check after extraction
        for c in Path('/content').rglob('Clear_Patches'):
            if len(list(c.glob('*.npy'))) > 0:
                cloudy = c.parent / 'Cloudy_Patches'
                return c, cloudy
    return None, None

CLEAR_DIR, CLOUDY_DIR = locate_dataset()

if CLEAR_DIR is not None:
    num_clear = len(list(CLEAR_DIR.glob('*.npy')))
    num_cloudy = len(list(CLOUDY_DIR.glob('*.npy')))
    print(f'\n[SUCCESS] Dataset Ready!')
    print(f'  Clear Patches Directory : {CLEAR_DIR} ({num_clear} patches)')
    print(f'  Cloudy Patches Directory: {CLOUDY_DIR} ({num_cloudy} patches)')
else:
    print('\n[!] Warning: Patches not detected yet.')
    print('To fix, simply upload your zip file (e.g. liss_iv_5k.zip) to Colab Files on the left panel, then re-run this cell.')


## Step 2: Mount Google Drive or Locate Dataset
Mount your Google Drive to load your pre-extracted patches, or extract a zipped patch folder:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set paths to where your patches are stored on Google Drive or locally
# Example: If stored in Google Drive: /content/drive/MyDrive/LISS-IV-Dataset/
DATA_DIR = Path("/content/drive/MyDrive/LISS-IV-Dataset")
CLEAR_DIR = DATA_DIR / "Clear_Patches"
CLOUDY_DIR = DATA_DIR / "Cloudy_Patches"

# If running directly with uploaded zip file:
if not CLEAR_DIR.exists():
    print(f"[!] {CLEAR_DIR} not found on Drive. Checking local /content/ ...")
    CLEAR_DIR = Path("/content/Data/Clear_Patches")
    CLOUDY_DIR = Path("/content/Data/Cloudy_Patches")

print(f"Clear directory:  {CLEAR_DIR}")
print(f"Cloudy directory: {CLOUDY_DIR}")


## Step 3: PyTorch Dataset with 4-Channel Guidance Tensor
Loads paired patches `(Cloudy, Clear)`, computes the cloud mask on the fly (or loads it), and packages the **4-channel guidance tensor** `[B2, B3, B4, Mask]`.


In [ ]:
class LISSPairDataset(Dataset):
    def __init__(self, clear_dir, cloudy_dir, indices=None, augment=True):
        self.clear_dir = Path(clear_dir)
        self.cloudy_dir = Path(cloudy_dir)
        self.augment = augment

        all_clears = sorted(list(self.clear_dir.glob("*.npy")))
        all_cloudies = sorted(list(self.cloudy_dir.glob("*.npy")))
        total = min(len(all_clears), len(all_cloudies))

        if indices is not None:
            self.clear_files = [all_clears[i] for i in indices]
            self.cloudy_files = [all_cloudies[i] for i in indices]
        else:
            self.clear_files = all_clears[:total]
            self.cloudy_files = all_cloudies[:total]

    def __len__(self):
        return len(self.clear_files)

    def __getitem__(self, idx):
        clear = np.load(self.clear_files[idx]).astype(np.float32)    # (3, 256, 256) in [0, 1]
        cloudy = np.load(self.cloudy_files[idx]).astype(np.float32)  # (3, 256, 256) in [0, 1]

        # Spatial Augmentations
        if self.augment:
            if np.random.rand() > 0.5:
                clear = np.flip(clear, axis=2).copy()
                cloudy = np.flip(cloudy, axis=2).copy()
            if np.random.rand() > 0.5:
                clear = np.flip(clear, axis=1).copy()
                cloudy = np.flip(cloudy, axis=1).copy()
            k = np.random.randint(0, 4)
            if k > 0:
                clear = np.rot90(clear, k=k, axes=(1, 2)).copy()
                cloudy = np.rot90(cloudy, k=k, axes=(1, 2)).copy()

        # Compute continuous cloud guidance mask via spectral whiteness & luminance
        mean_int = (cloudy[0] + cloudy[1] + cloudy[2]) / 3.0
        spread = (np.abs(cloudy[0] - mean_int) + np.abs(cloudy[1] - mean_int) + np.abs(cloudy[2] - mean_int)) / 3.0
        whiteness = 1.0 - np.clip(spread / 0.22, 0.0, 1.0)
        brightness = np.clip((mean_int - 0.40) / 0.60, 0.0, 1.0)
        mask = np.clip(brightness * whiteness, 0.0, 1.0)
        mask_channel = np.expand_dims(mask, axis=0)  # (1, 256, 256)

        # 4-channel input: [B2, B3, B4, Mask]
        guidance_input = np.concatenate([cloudy, mask_channel], axis=0)

        # Scale satellite bands [0, 1] -> [-1, 1] for Tanh Generator
        clear_scaled = (clear * 2.0) - 1.0
        guidance_scaled = guidance_input.copy()
        guidance_scaled[:3] = (guidance_scaled[:3] * 2.0) - 1.0  # Bands to [-1, 1], mask stays [0, 1]

        return torch.from_numpy(guidance_scaled), torch.from_numpy(clear_scaled)

print("[*] LISSPairDataset class loaded successfully.")


## Step 4: Adapted Pix2Pix Model Architecture (4-Channel U-Net & PatchGAN)
Strictly adapted from Isola et al. (CVPR 2017) to process 4 input channels:


In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=not normalize)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_ch, affine=True))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.ReLU(inplace=True)
        ]
        if dropout > 0.0:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.block(x)
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """4-Channel Input [B2, B3, B4, Mask] -> 3-Channel Output [B2, B3, B4]."""
    def __init__(self, in_channels=4, out_channels=3, base_filters=64):
        super().__init__()
        f = base_filters
        # Encoder
        self.d1 = DownBlock(in_channels, f, normalize=False)
        self.d2 = DownBlock(f, f * 2)
        self.d3 = DownBlock(f * 2, f * 4)
        self.d4 = DownBlock(f * 4, f * 8)
        self.d5 = DownBlock(f * 8, f * 8)
        self.d6 = DownBlock(f * 8, f * 8)
        self.d7 = DownBlock(f * 8, f * 8)
        self.d8 = DownBlock(f * 8, f * 8, normalize=False)
        # Decoder with skip connections
        self.u1 = UpBlock(f * 8, f * 8, dropout=0.5)
        self.u2 = UpBlock(f * 16, f * 8, dropout=0.5)
        self.u3 = UpBlock(f * 16, f * 8, dropout=0.5)
        self.u4 = UpBlock(f * 16, f * 8)
        self.u5 = UpBlock(f * 16, f * 4)
        self.u6 = UpBlock(f * 8, f * 2)
        self.u7 = UpBlock(f * 4, f)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(f * 2, out_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        e1 = self.d1(x)
        e2 = self.d2(e1)
        e3 = self.d3(e2)
        e4 = self.d4(e3)
        e5 = self.d5(e4)
        e6 = self.d6(e5)
        e7 = self.d7(e6)
        e8 = self.d8(e7)
        u1 = self.u1(e8, e7)
        u2 = self.u2(u1, e6)
        u3 = self.u3(u2, e5)
        u4 = self.u4(u3, e4)
        u5 = self.u5(u4, e3)
        u6 = self.u6(u5, e2)
        u7 = self.u7(u6, e1)
        return self.final(u7)


class PatchGANDiscriminator(nn.Module):
    """70x70 Markovian PatchGAN: Input 4 + 3 = 7 channels."""
    def __init__(self, in_channels=7, base_filters=64):
        super().__init__()
        f = base_filters
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, f, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(f, f * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(f * 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(f * 2, f * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(f * 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(f * 4, f * 8, kernel_size=4, stride=1, padding=1, bias=False),
            nn.InstanceNorm2d(f * 8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(f * 8, 1, kernel_size=4, stride=1, padding=1)
        )

    def forward(self, cond, target):
        return self.net(torch.cat([cond, target], dim=1))

print("[*] UNetGenerator and PatchGANDiscriminator defined.")


## Step 5: Remote Sensing Loss Functions (SAM & NDVI Consistency)
These loss functions preserve Near-Infrared agricultural reflectance and prevent color shifting:


In [ ]:
class SpectralAngleMapperLoss(nn.Module):
    """Enforces radiometric vector fidelity across B2, B3, B4."""
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        p = (pred + 1.0) * 0.5
        t = (target + 1.0) * 0.5
        dot = torch.sum(p * t, dim=1)
        norm_p = torch.sqrt(torch.sum(p ** 2, dim=1) + self.eps)
        norm_t = torch.sqrt(torch.sum(t ** 2, dim=1) + self.eps)
        cos_theta = torch.clamp(dot / (norm_p * norm_t), -1.0 + self.eps, 1.0 - self.eps)
        return torch.mean(torch.acos(cos_theta))


class NDVIConsistencyLoss(nn.Module):
    """Protects crop and forest biological indices: NDVI = (NIR - Red) / (NIR + Red)."""
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, pred, target):
        p = (pred + 1.0) * 0.5
        t = (target + 1.0) * 0.5
        pred_red, pred_nir = p[:, 1:2, :, :], p[:, 2:3, :, :]
        tgt_red, tgt_nir = t[:, 1:2, :, :], t[:, 2:3, :, :]
        ndvi_p = (pred_nir - pred_red) / (pred_nir + pred_red + self.eps)
        ndvi_t = (tgt_nir - tgt_red) / (tgt_nir + tgt_red + self.eps)
        return F.l1_loss(ndvi_p, ndvi_t)

print("[*] SAM Loss and NDVI Consistency Loss initialized.")


## Step 6: Training Setup, Optimizers & Mixed Precision


In [ ]:
# Initialize Models
generator = UNetGenerator(in_channels=4, out_channels=3).to(device)
discriminator = PatchGANDiscriminator(in_channels=7).to(device)

# Weights initialization (Pix2Pix standard: normal mean=0, std=0.02)
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif isinstance(m, nn.InstanceNorm2d) and m.weight is not None:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

generator.apply(init_weights)
discriminator.apply(init_weights)

# Loss functions
criterion_gan = nn.MSELoss()  # LSGAN formulation for stable training
criterion_l1 = nn.L1Loss()
criterion_sam = SpectralAngleMapperLoss()
criterion_ndvi = NDVIConsistencyLoss()

# Loss Weights
LAMBDA_L1 = 100.0
LAMBDA_SAM = 10.0
LAMBDA_NDVI = 5.0

# Optimizers
lr = 0.0002
optimizer_G = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

# Automatic Mixed Precision for 2x faster training on T4 GPU
scaler_G = torch.cuda.amp.GradScaler()
scaler_D = torch.cuda.amp.GradScaler()

print("[*] Training pipeline ready on:", device)


## Step 7: Side-by-Side Visual Inspection Helper
Renders False Color Composite (FCC: NIR $\to$ R, Red $\to$ G, Green $\to$ B) at the end of each epoch:


In [ ]:
def to_fcc_display(tensor_3ch):
    """Converts tensor (3, H, W) in [-1, 1] into FCC [B4, B3, B2] image in [0, 1]."""
    arr = tensor_3ch.detach().cpu().numpy()
    arr = (arr + 1.0) * 0.5  # [-1, 1] -> [0, 1]
    b2, b3, b4 = arr[0], arr[1], arr[2]
    fcc = np.stack([b4, b3, b2], axis=-1)
    return np.clip(fcc, 0.0, 1.0)


def visualize_progress(gen, val_loader, epoch):
    gen.eval()
    with torch.no_grad():
        inputs, targets = next(iter(val_loader))
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = gen(inputs)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(to_fcc_display(inputs[0, :3]))
    axes[0].set_title("Cloudy Input (FCC: NIR, Red, Green)")
    axes[0].axis("off")

    axes[1].imshow(to_fcc_display(outputs[0]))
    axes[1].set_title(f"Pix2Pix Generated Output (Epoch {epoch})")
    axes[1].axis("off")

    axes[2].imshow(to_fcc_display(targets[0]))
    axes[2].set_title("Ground Truth Clear (Target)")
    axes[2].axis("off")

    plt.suptitle(f"Epoch {epoch} Inpainting Comparison", fontsize=14)
    plt.tight_layout()
    plt.show()
    gen.train()


## Step 8: The Training Loop
Executes training on GPU, plots validation previews every epoch, and saves `generator_best.pth`:


In [ ]:
# Set batch size & epochs (Change EPOCHS as desired, e.g., 20 - 50)
BATCH_SIZE = 16
EPOCHS = 20
SAVE_DIR = Path('/content/checkpoints')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Ensure CLEAR_DIR is found (self-healing)
if 'CLEAR_DIR' not in globals() or CLEAR_DIR is None or len(list(CLEAR_DIR.glob('*.npy'))) == 0:
    CLEAR_DIR, CLOUDY_DIR = locate_dataset()

assert CLEAR_DIR is not None, (
    'No patches found! Please upload your liss_iv_5k.zip to Colab files on the left panel '
    'or run Step 2 to locate and extract your dataset.'
)

# Create DataLoader (using available patches)
all_clears = sorted(list(CLEAR_DIR.glob('*.npy')))
total = len(all_clears)
assert total > 0, f'No .npy patches found inside {CLEAR_DIR}!'

val_count = min(500, int(total * 0.10))
train_indices = list(range(total - val_count))
val_indices = list(range(total - val_count, total))

train_ds = LISSPairDataset(CLEAR_DIR, CLOUDY_DIR, indices=train_indices, augment=True)
val_ds = LISSPairDataset(CLEAR_DIR, CLOUDY_DIR, indices=val_indices, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

print(f'[*] Training on {len(train_ds)} patches | Validating on {len(val_ds)} patches.')

best_l1_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    generator.train()
    discriminator.train()
    total_g_loss = 0.0
    total_d_loss = 0.0
    total_l1_loss = 0.0
    start_time = time.time()
    
    for i, batch in enumerate(train_loader):
        cloudy = batch['cloudy'].to(device, non_blocking=True)
        clear = batch['clear'].to(device, non_blocking=True)
        guidance = batch['guidance'].to(device, non_blocking=True)
        
        # ------------------------------------------------------------------
        # Train Discriminator: maximize log(D(x, y)) + log(1 - D(x, G(x)))
        # ------------------------------------------------------------------
        opt_D.zero_grad()
        with torch.cuda.amp.autocast(enabled=True):
            # Real loss
            pred_real = discriminator(guidance, clear)
            loss_d_real = criterion_adv(pred_real, torch.ones_like(pred_real))
            
            # Fake loss
            fake_clear = generator(guidance)
            pred_fake = discriminator(guidance, fake_clear.detach())
            loss_d_fake = criterion_adv(pred_fake, torch.zeros_like(pred_fake))
            
            loss_D = (loss_d_real + loss_d_fake) * 0.5
            
        scaler_D.scale(loss_D).backward()
        scaler_D.step(opt_D)
        scaler_D.update()
        
        # ------------------------------------------------------------------
        # Train Generator: fool D + L1 + SAM + NDVI
        # ------------------------------------------------------------------
        opt_G.zero_grad()
        with torch.cuda.amp.autocast(enabled=True):
            pred_fake_for_g = discriminator(guidance, fake_clear)
            loss_g_adv = criterion_adv(pred_fake_for_g, torch.ones_like(pred_fake_for_g))
            loss_g_l1 = criterion_l1(fake_clear, clear) * 100.0
            loss_g_sam = criterion_sam(fake_clear, clear) * 5.0
            loss_g_ndvi = criterion_ndvi(fake_clear, clear) * 10.0
            
            loss_G = loss_g_adv + loss_g_l1 + loss_g_sam + loss_g_ndvi
            
        scaler_G.scale(loss_G).backward()
        scaler_G.step(opt_G)
        scaler_G.update()
        
        total_g_loss += loss_G.item()
        total_d_loss += loss_D.item()
        total_l1_loss += (loss_g_l1.item() / 100.0)
        
    elapsed = time.time() - start_time
    avg_g = total_g_loss / len(train_loader)
    avg_d = total_d_loss / len(train_loader)
    avg_l1 = total_l1_loss / len(train_loader)
    
    print(f'[Epoch {epoch:02d}/{EPOCHS:02d}] Duration: {elapsed:5.1f}s | G_Loss: {avg_g:.4f} | D_Loss: {avg_d:.4f} | L1: {avg_l1:.4f}')
    
    # Save best checkpoint
    if avg_l1 < best_l1_loss:
        best_l1_loss = avg_l1
        save_path = SAVE_DIR / 'generator_best.pth'
        torch.save(generator.state_dict(), save_path)
        print(f'   [*] New best model saved -> {save_path}')
        
    # Visual preview every 2 epochs (or epoch 1)
    if epoch == 1 or epoch % 2 == 0:
        show_sample_preview(generator, val_ds, sample_idx=0)


## Step 9: Download Trained Weights to Your Computer


In [ ]:
from google.colab import files

# Download the best generator weights directly to your laptop
files.download(str(SAVE_DIR / "generator_best.pth"))
print("[*] Download initiated.")
